# 🔬 Replication & Transferability Study: Jamil Khan Emon et al. (CMC 2025)
### Paper: Harmonization of Heart Disease Dataset for Accurate Diagnosis: A Machine Learning Approach Enhanced by Feature Engineering
**Authors:** Jamil Khan Emon et al. (Computers, Materials & Continua, 2025)
**GitHub Source:** [https://github.com/JamilKhanEmon/Heart-Disease-](https://github.com/JamilKhanEmon/Heart-Disease-)

This notebook demonstrates the exact methodological structure and code flow from  and  applied to FinSight Banking Intelligence datasets.

In [ ]:
# Step 2: Importing Libraries
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

print("Libraries successfully imported!")

In [ ]:
# Step 3: Load Banking Dataset (Bank Marketing Prospective or Customer Churn)
from ai.data.cleaner import DatasetCleaner
from ai.data.ingestion import load_raw_dataset

raw_df = load_raw_dataset("customer_churn")
cleaner = DatasetCleaner("customer_churn")
clean_df, _ = cleaner.clean_customer_churn(raw_df)

df = clean_df.copy()
target_col = "Exited"
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
# Cell 15 Replication: Correlation Values for Two-Variable Combinations (Highest to Lowest)
num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != target_col]
pairs = []
for i in range(len(num_cols)):
    for j in range(i + 1, len(num_cols)):
        pairs.append((num_cols[i], num_cols[j]))

correlation_results = []
for var1, var2 in pairs:
    corr_val = df[var1].corr(df[var2])
    correlation_results.append((f"{var1} * {var2}", corr_val, abs(corr_val)))

correlation_df = pd.DataFrame(correlation_results, columns=["Variable Pair", "Correlation Value", "Abs Value"])
correlation_df_sorted = correlation_df.sort_values(by="Abs Value", ascending=False).drop(columns=["Abs Value"])

print("Correlation Values for Two-Variable Combinations (Highest to Lowest):")
correlation_df_sorted.head(10)

In [ ]:
# Cell 17 Replication: Performing Chi-Squared Test for Categorical Variables
cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
print("Chi-Squared Test Results for Categorical Features:")
for i in range(len(cat_cols)):
    for j in range(i + 1, len(cat_cols)):
        v1, v2 = cat_cols[i], cat_cols[j]
        contingency = pd.crosstab(df[v1], df[v2])
        chi2, p, dof, _ = chi2_contingency(contingency)
        sig = "Significant (p < 0.05)" if p < 0.05 else "Not significant"
        print(f"{v1} vs {v2} -> Chi2: {chi2:.2f}, p-val: {p:.4e} -> {sig}")

In [ ]:
# Cell 18 Replication: Feature Engineering - Creating Multiplicative Interaction Terms
print("Feature Engineering - Creating New Interaction Features")
top_pairs = [tuple(p.split(" * ")) for p in correlation_df_sorted["Variable Pair"].head(6)]
for v1, v2 in top_pairs:
    new_col = f"{v1}_{v2}"
    df[new_col] = df[v1] * df[v2]

print(f"New Total Features: {df.shape[1]}")
df.head()

In [ ]:
# Cell 19 Replication: Feature Scaling Applied (StandardScaler)
active_num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != target_col]
scaler = StandardScaler()
df[active_num_cols] = scaler.fit_transform(df[active_num_cols])
print("Feature Scaling Applied:")
df[active_num_cols].head()

In [ ]:
# Cell 20 Replication: Train-Test Split (70-30%)
X = df[active_num_cols]
y = df[target_col].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
print(f"Training Set Shape: {X_train.shape}")
print(f"Testing Set Shape: {X_test.shape}")

In [ ]:
# Cell 21 Replication: Applying PCA for Dimensionality Reduction (95% Variance Retention)
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"Original number of features: {X_train.shape[1]}")
print(f"Reduced number of features after PCA: {X_train_pca.shape[1]}")
print(f"Cumulative Explained Variance: {np.sum(pca.explained_variance_ratio_):.4f}")

In [ ]:
# Cell 22 Replication: Model Training and Run Time Tracking
models = {
    "RandomForest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=4),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, eval_metric="logloss", n_jobs=4),
}

best_models = {}
run_times = {}

for model_name, model in models.items():
    start_time = time.time()
    model.fit(X_train_pca, y_train)
    elapsed = time.time() - start_time
    best_models[model_name] = model
    run_times[model_name] = elapsed
    print(f"{model_name} Run Time: {elapsed:.2f} seconds")

In [ ]:
# Cell 23 Replication: Creating Soft Voting Classifier
voting_clf = VotingClassifier(
    estimators=[(name, model) for name, model in best_models.items()],
    voting="soft"
)
start_time = time.time()
voting_clf.fit(X_train_pca, y_train)
voting_time = time.time() - start_time
print(f"Voting Classifier Training Run Time: {voting_time:.2f} seconds")

In [ ]:
# Cell 24 & 26 Replication: Performance Metrics, Sensitivity, Specificity & Confusion Matrix
y_pred_voting = voting_clf.predict(X_test_pca)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_voting).ravel()
voting_acc = accuracy_score(y_test, y_pred_voting)
precision_voting = precision_score(y_test, y_pred_voting)
recall_voting = tp / (tp + fn)
specificity_voting = tn / (tn + fp)
f1_voting = f1_score(y_test, y_pred_voting)

print("\nVoting Classifier Results:")
print(f"Accuracy: {voting_acc:.4f}")
print(f"Precision: {precision_voting:.4f}")
print(f"Sensitivity (Recall): {recall_voting:.4f}")
print(f"Specificity: {specificity_voting:.4f}")
print(f"F1 Score: {f1_voting:.4f}")

disp_voting = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_test, y_pred_voting), display_labels=[0, 1])
disp_voting.plot(cmap="Blues")
plt.title("Voting Classifier Confusion Matrix")
plt.show()